# Greek Mountain Tea data lineage

Plant ID: PLANT047; scientific name: Sideritis scardica. This notebook checks the compound flow into AI and disease analysis files.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
PLANT_ID = 'PLANT047'
FILES = {
    'plant catalog': 'data/raw/medicinal_plants.csv',
    'compound source': 'data/raw/new_plant_compounds_pubchem.csv',
    'ChEMBL retry': 'data/processed/new_plant_compounds_chembl_retry.csv',
    'RDKit features': 'data/processed/new_plant_rdkit_features.csv',
    'prediction-ready': 'data/processed/new_plant_ai_prediction_ready.csv',
    'new AI predictions': 'data/processed/new_plant_ai_predictions.csv',
    'generic AI predictions': 'data/processed/plant_ai_predictions.csv',
    'human disease predictions': 'data/processed/plant_ai_human_predictions.csv',
    'T2D predictions': 'data/processed/t2d_plant_predictions.csv',
}

def plant_rows(path):
    df = pd.read_csv(ROOT / path, low_memory=False)
    return df[df['Plant_ID'].astype(str).str.strip() == PLANT_ID].copy()

summary = []
for stage, path in FILES.items():
    df = plant_rows(path)
    summary.append({
        'stage': stage,
        'rows': len(df),
        'compounds': df['Compound_Name'].nunique() if 'Compound_Name' in df else None,
        'nonempty ChEMBL IDs': df['ChEMBL_ID'].notna().sum() if 'ChEMBL_ID' in df else None,
        'targets': df['UniProt_ID'].nunique() if 'UniProt_ID' in df else None,
    })
pd.DataFrame(summary)

## Findings

- The plant is registered as Greek Mountain Tea (Sideritis scardica).
- One compound has PubChem SMILES and RDKit features, but ChEMBL lookup is NOT_FOUND, leaving ChEMBL_ID blank.
- The generic model produced 382 target rows, and T2D has 6 prediction rows. Their ChEMBL IDs are blank, so T2D Compound_Count is 0.
- The non-T2D website reads plant_ai_human_predictions.csv, which currently has no PLANT047 rows.